# App Review Sentiment Analyzer
**Author:** Ayaanika Kakar

**Tools:** Python · Pandas · NLTK · TextBlob · Google Colab

---

### What this project does
This notebook simulates a **PM user research workflow** at scale. Instead of reading hundreds of reviews manually, we use NLP to:
- Clean and normalize raw, messy user-generated text
- Score each review's sentiment automatically
- Extract the top words driving positive vs. negative feedback
- Generate a product insight report that could directly inform a backlog prioritization decision

**Real-world application:** Feed 10,000+ App Store or Play Store reviews into this pipeline to understand what users love and hate — without a single manual read.

---
## Step 0: Install & Import Libraries

In [ ]:
# Run this cell first — installs TextBlob and downloads NLTK data
!pip install textblob --quiet

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('Setup complete!')

In [ ]:
import pandas as pd
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob

---
## Step 1: Load the Data

We use **460,256 real TikTok Google Play Store reviews** sourced from Kaggle. In a real PM context, you'd pull these directly from the App Store API, a Play Store scraper, or a data pipeline — the NLP workflow is identical regardless of scale.

In [ ]:
df = pd.read_csv('/tiktok_reviews.csv')
df = df.rename(columns={'content': 'review'})
df = df.dropna(subset=['review'])
df['review'] = df['review'].astype(str)
print(f'Loaded {len(df):,} reviews')
df.head()

---
## Step 2: Clean the Text

Raw reviews are messy — emojis, punctuation, numbers, inconsistent casing. We normalize all of it **before** any analysis so our model treats `"LOVE"` and `"love"` as the same word.

The regex `[^a-z\s]` means: **remove anything that is NOT a lowercase letter or a space.**

In [ ]:
def clean_text(text):
    text = text.lower()                          # 1. lowercase everything
    text = re.sub(r'[^a-z\s]', '', text)         # 2. strip punctuation, numbers, emojis
    text = re.sub(r'\s+', ' ', text).strip()     # 3. collapse extra whitespace
    return text

df['clean'] = df['review'].apply(clean_text)

# Show before/after on two real reviews
print('--- Example 1 ---')
print('BEFORE:', df['review'].iloc[2])
print('AFTER: ', df['clean'].iloc[2])
print()
print('--- Example 2 ---')
print('BEFORE:', df['review'].iloc[17])
print('AFTER: ', df['clean'].iloc[17])

---
## Step 3: Tokenize & Remove Stopwords

**Tokenization** splits a sentence into individual word units called *tokens*.

**Stopwords** are ultra-common words (`"the"`, `"is"`, `"and"`, `"a"`) that appear everywhere but carry no useful signal. We remove them so only **meaningful words** survive.

**Lemmatization** reduces words to their root form: `"crashing"` → `"crash"`, `"ads"` → `"ad"`. This prevents the same concept from splitting across multiple words.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def tokenize_and_filter(text):
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(w)
        for w in tokens
        if w not in stop_words and len(w) > 2
    ]
    return tokens

df['tokens'] = df['clean'].apply(tokenize_and_filter)

# Show sample tokens from 2 random reviews
sample = df[['review', 'tokens']].sample(2, random_state=7)
for _, row in sample.iterrows():
    print('Review:', row['review'][:120])
    print('Tokens:', row['tokens'])
    print()

---
## Step 4: Sentiment Analysis with TextBlob

TextBlob assigns each review a **polarity score**:
- `+1.0` = very positive
- `0.0` = neutral
- `-1.0` = very negative

We use a threshold approach to classify each review:

In [ ]:
def get_sentiment(text):
    score = TextBlob(text).sentiment.polarity
    if score > 0.1:
        label = 'Positive'
    elif score < -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'
    return label, round(score, 3)

df[['sentiment', 'polarity']] = df['clean'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

print('Sentiment breakdown:')
counts = df['sentiment'].value_counts()
for label, count in counts.items():
    print(f'  {label}: {count:,} ({count/len(df)*100:.1f}%)')
print(f'\nAverage polarity score: {df["polarity"].mean():.3f}')
print()
print('Sample scored reviews:')
print(df[['review', 'sentiment', 'polarity']].sample(6, random_state=42).to_string(index=False))

---
## Step 5: Visualize Sentiment Distribution

In [ ]:
sentiment_counts = df['sentiment'].value_counts()
color_map = {'Positive': '#4CAF50', 'Negative': '#F44336', 'Neutral': '#FF9800'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('TikTok App Review Sentiment Analysis', fontsize=14, fontweight='bold', y=1.01)

# --- Bar chart ---
bar_colors = [color_map.get(s, '#888') for s in sentiment_counts.index]
bars = axes[0].bar(sentiment_counts.index, sentiment_counts.values,
                   color=bar_colors, edgecolor='white', linewidth=0.8)
axes[0].set_title(f'Sentiment Distribution (n={len(df):,})')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_ylim(0, max(sentiment_counts.values) * 1.12)
for bar, val in zip(bars, sentiment_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=9)

# --- Polarity histogram ---
axes[1].hist(df['polarity'], bins=14, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Neutral boundary')
axes[1].axvline(df['polarity'].mean(), color='gold', linestyle='-', linewidth=1.5,
                label=f'Mean = {df["polarity"].mean():.2f}')
axes[1].set_title('Polarity Score Distribution')
axes[1].set_xlabel('Polarity  (−1 = very negative  |  +1 = very positive)')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 6: Top Keywords by Sentiment

Which words dominate positive vs. negative reviews? This is where signal becomes **product insight**.

In [ ]:
def top_words(sentiment_label, n=10):
    all_tokens = df[df['sentiment'] == sentiment_label]['tokens'].explode()
    return Counter(all_tokens).most_common(n)

pos_words = top_words('Positive')
neg_words = top_words('Negative')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pw_words, pw_counts = zip(*pos_words)
axes[0].barh(pw_words[::-1], pw_counts[::-1], color='#4CAF50', edgecolor='white')
axes[0].set_title('Top Words — Positive Reviews', fontweight='bold')
axes[0].set_xlabel('Frequency')

nw_words, nw_counts = zip(*neg_words)
axes[1].barh(nw_words[::-1], nw_counts[::-1], color='#F44336', edgecolor='white')
axes[1].set_title('Top Words — Negative Reviews', fontweight='bold')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('top_keywords.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top Positive Words:', [w for w, _ in pos_words])
print('Top Negative Words:', [w for w, _ in neg_words])

---
## Step 7: Product Insight Report

This is the PM layer — translating raw NLP output into **named themes** that map directly to engineering tickets or backlog items.

In [ ]:
complaint_themes = {
    'App Stability  (crashes / freezes)': ['crash', 'crashing', 'freezing', 'freeze', 'bug', 'broken'],
    'Ad Overload': ['ad', 'ads', 'sponsored', 'advertisement'],
    'Privacy / Data Concerns': ['privacy', 'data', 'trust'],
    'Performance  (battery / loading)': ['battery', 'loading', 'slow', 'performance', 'heavy'],
    'Algorithm Issues': ['algorithm', 'content', 'recommendation', 'feed'],
    'Support & Notifications': ['support', 'notification', 'shadowban', 'alert'],
}

praise_themes = {
    'Personalization / Algorithm': ['algorithm', 'recommendation', 'personalized', 'foryou', 'discover'],
    'Content Quality': ['content', 'creator', 'video', 'entertainment', 'trend'],
    'Features  (live, duet, edit)': ['live', 'duet', 'editing', 'feature', 'tool'],
    'Overall UX': ['ux', 'smooth', 'intuitive', 'design', 'easy'],
}

neg_reviews = df[df['sentiment'] == 'Negative']
pos_reviews = df[df['sentiment'] == 'Positive']

print('=' * 58)
print('         PRODUCT INSIGHT REPORT')
print(f'         Based on {len(df):,} real TikTok reviews')
print('=' * 58)

print('\n TOP PAIN POINTS  (from negative reviews)\n')
for theme, keywords in complaint_themes.items():
    count = neg_reviews['tokens'].apply(
        lambda t: any(k in t for k in keywords)
    ).sum()
    if count > 0:
        pct = round(count / len(neg_reviews) * 100, 1)
        bar = '█' * min(count // 50, 40)
        print(f'  {theme:<42} {count:>6,} reviews ({pct}%)')
        print(f'  {bar}')
        print()

print('\n WHAT USERS LOVE  (from positive reviews)\n')
for theme, keywords in praise_themes.items():
    count = pos_reviews['tokens'].apply(
        lambda t: any(k in t for k in keywords)
    ).sum()
    if count > 0:
        pct = round(count / len(pos_reviews) * 100, 1)
        bar = '█' * min(count // 500, 40)
        print(f'  {theme:<42} {count:>6,} reviews ({pct}%)')
        print(f'  {bar}')
        print()

print('\n PM RECOMMENDATION')
print('  Stability and ad frequency are the two highest-volume pain points.')
print('  Prioritize crash fixes (P0) and A/B test reduced ad frequency')
print('  before shipping new features — users cite algorithm as a key')
print('  retention driver, so protecting that experience is critical.')

---
## Summary

| Step | Tool | What it does |
|------|------|--------------|
| Clean | `re`, `pandas` | Lowercase, strip punctuation, collapse whitespace |
| Tokenize | `NLTK word_tokenize` | Split sentences into individual word units |
| Filter | `NLTK stopwords` | Remove noise words; lemmatize to root form |
| Score | `TextBlob` | Assign polarity (−1 to +1) to each review |
| Analyze | `Counter`, `matplotlib` | Surface top keywords and complaint themes |